In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path    

In [ ]:
# conecta a la base de datos mercado inmobiliario.db y carga la tabla tablon_analitico y cargala en un dataframe llamado df
import sqlite3

conn = sqlite3.connect('../datos/intermedios/analisis_inmobiliario.db')
df = pd.read_sql('SELECT * FROM tablon_analitico', conn)
conn.close()

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df['precio_noche'].describe()

In [ ]:
rango_precios = (df['precio_noche'].min(), df['precio_noche'].max())
rango_precios

In [ ]:
precio_mediana_distrito = df.groupby('distrito')['precio_noche'].median().sort_values(ascending=False)
precio_mediana_distrito

In [ ]:
precio_mediana_barrio = df.groupby('neighbourhood')['precio_noche'].median().sort_values(ascending=False)
precio_mediana_barrio

In [ ]:
factores_precio = df.corr(numeric_only=True)['precio_noche'].sort_values(ascending=False)
factores_precio

In [ ]:
competencia_barrio = df.groupby('neighbourhood').size()
competencia_barrio

In [ ]:
precio_vs_competencia = df.groupby('neighbourhood').agg({'precio_noche':'median', 'name':'count'}).rename(columns={'name':'num_inmuebles'})
precio_vs_competencia

In [ ]:
precio_por_tipo = df.groupby('room_type')['precio_noche'].median()
precio_por_tipo

In [ ]:
ocupacion_mediana = df['estimated_occupancy_l365d'].median()
ocupacion_mediana

In [ ]:
ocupacion_mediana_distrito = df.groupby('distrito')['estimated_occupancy_l365d'].median().sort_values(ascending=False)
ocupacion_mediana_distrito

In [ ]:
ocupacion_mediana_barrio = df.groupby('neighbourhood')['estimated_occupancy_l365d'].median().sort_values(ascending=False)
ocupacion_mediana_barrio

In [ ]:
prob_ocupacion_distrito = df.groupby(['distrito', 'estimated_occupancy_l365d']).size().groupby(level=0).apply(lambda x: x / x.sum())
prob_ocupacion_distrito

# Analisis de Minicubo

In [ ]:
df.columns

In [ ]:
dimensiones = ['distrito', 'neighbourhood', 'room_type', 'bedrooms_disc', 'accommodates_disc', 'beds_disc']

metricas = ['precio_noche_total', 'estimated_occupancy_l365d', 'ingreso_anual', 'coste_adquisicion', 'margen_bruto']

In [ ]:
minicubo = df[dimensiones + metricas]
minicubo

In [ ]:
# Paso 2: Pasar a transaccional las dimensiones
minicubo_melt = minicubo.melt(id_vars=metricas, value_vars=dimensiones, var_name='dimension', value_name='valor')
minicubo_melt

In [ ]:
# Paso 3: Agregar las métricas por "dimension" y "valor" con funciones deseadas (conteo y mediana)
minicubo_agg = minicubo_melt.groupby(['dimension', 'valor']).agg(
    mediana_precio_noche_total=('precio_noche_total', 'median'),
    mediana_ocupacion=('estimated_occupancy_l365d', 'median'),
    mediana_ingreso_anual=('ingreso_anual', 'median'),
    mediana_coste_adquisicion=('coste_adquisicion', 'median'),
    mediana_margen_bruto=('margen_bruto', 'median')
).reset_index()
minicubo_agg

In [ ]:
# Filtramos solo la métrica de interés
metrica = 'mediana_precio_noche_total'

# Obtenemos las dimensiones únicas
dimensiones_unicas = minicubo_agg['dimension'].unique()

for dimension in dimensiones_unicas:
    datos = minicubo_agg[minicubo_agg['dimension'] == dimension].sort_values(metrica, ascending=False)
    plt.figure(figsize=(10, 4))
    plt.bar(datos['valor'].astype(str), datos[metrica])
    plt.title(f"{dimension} - Mediana precio_noche_total")
    plt.ylabel("Mediana precio_noche_total (€)")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Elimina filas de neighbourhood 'Fuentelareina'
df = df[df['neighbourhood'] != 'Fuentelareina']

In [ ]:
metrica = 'mediana_margen_bruto'
dimensiones_unicas = minicubo_agg['dimension'].unique()

for dimension in dimensiones_unicas:
    datos = minicubo_agg[minicubo_agg['dimension'] == dimension].sort_values(metrica, ascending=False)
    plt.figure(figsize=(10, 4))
    plt.bar(datos['valor'].astype(str), datos[metrica])
    plt.title(f"{dimension} - Mediana margen_bruto")
    plt.ylabel("Mediana margen_bruto (%)")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
metrica = 'mediana_coste_adquisicion'
dimensiones_unicas = minicubo_agg['dimension'].unique()

for dimension in dimensiones_unicas:
    datos = minicubo_agg[minicubo_agg['dimension'] == dimension].sort_values(metrica, ascending=False)
    plt.figure(figsize=(10, 4))
    plt.bar(datos['valor'].astype(str), datos[metrica])
    plt.title(f"{dimension} - Mediana coste_adquisicion")
    plt.ylabel("Mediana coste_adquisicion (€)")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
metrica = 'mediana_ingreso_anual'
dimensiones_unicas = minicubo_agg['dimension'].unique()

for dimension in dimensiones_unicas:
    datos = minicubo_agg[minicubo_agg['dimension'] == dimension].sort_values(metrica, ascending=False)
    plt.figure(figsize=(10, 4))
    plt.bar(datos['valor'].astype(str), datos[metrica])
    plt.title(f"{dimension} - Mediana ingreso_anual")
    plt.ylabel("Mediana ingreso_anual (€)")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

# Análisis de Insights

- **bedrooms**: de 2 a 3  
- **accommodates**: más de 3  
- **beds**: más de 2  
- **distritos**: Barajas, Centro, Arganzuela, Villa de Vallecas  

> Podemos obtener rentabilidades de hasta un **40%**.

In [ ]:
# Filtrado según las condiciones indicadas
filtro = (
    (df['bedrooms'].isin([2, 3])) &
    (df['accommodates'] >= 3) &
    (df['beds'] >= 2) &
    (df['distrito'].isin(['Barajas', 'Centro', 'Arganzuela', 'Villa de Vallecas']))
)

df_filtrado = df[filtro]
df_filtrado

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df_filtrado['margen_bruto'], bins=30, color='skyblue', edgecolor='black')
plt.title('Histograma del margen bruto (registros filtrados)')
plt.xlabel('Margen bruto')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_filtrado['estimated_occupancy_l365d'], df_filtrado['margen_bruto'], alpha=0.7)
plt.xlabel('Ocupación estimada últimos 365 días')
plt.ylabel('Margen bruto')
plt.title('Margen bruto vs Ocupación (registros filtrados)')

# Dibuja el recuadro de la zona de interés
plt.axhline(7, color='red', linestyle='--', linewidth=1)
plt.axvline(200, color='red', linestyle='--', linewidth=1)
plt.gca().add_patch(
    plt.Rectangle(
        (200, 7),                # esquina inferior izquierda
        df_filtrado['estimated_occupancy_l365d'].max() - 200,  # ancho
        df_filtrado['margen_bruto'].max() - 7,                 # alto
        fill=False, edgecolor='black', linewidth=2, linestyle='--'
    )
)

plt.tight_layout()
plt.show()

# Análisis de Ocupación

#### Insights análisis ocupación (Inmuebles con mayor ocupación):

- 4 o más habitaciones 
- Con 5 o más personas   
- Especial interes en Barajas, es posible comprar inmuebles con ocupación superior a 200 días y rentabilidad superior al 10% por menos de 200.000

In [ ]:
metrica = 'mediana_ocupacion'
dimensiones_unicas = minicubo_agg['dimension'].unique()

for dimension in dimensiones_unicas:
    datos = minicubo_agg[minicubo_agg['dimension'] == dimension].sort_values(metrica, ascending=False)
    plt.figure(figsize=(10, 4))
    plt.bar(datos['valor'].astype(str), datos[metrica])
    plt.title(f"{dimension} - Mediana estimated_occupancy_l365d")
    plt.ylabel("Mediana estimated_occupancy_l365d (días)")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Crear un data frame con los registros de barajas
df_barajas = df[df['distrito'] == 'Barajas']

# Crear un scatter plot de coste adquisición vs margen bruto y la ocupación estimada como puntos de colores
plt.figure(figsize=(8, 5))
scatter = plt.scatter(df_barajas['coste_adquisicion'], df_barajas['margen_bruto'], c=df_barajas['estimated_occupancy_l365d'], cmap='viridis', alpha=0.7)
plt.xlabel('Coste de adquisición (€)')
plt.ylabel('Margen bruto (€)')
plt.title('Coste de adquisición vs Margen bruto (Barajas)')
cbar = plt.colorbar(scatter)
cbar.set_label('Ocupación estimada (días)')
plt.show()

In [ ]:
# Copia el DataFrame original
df_filtrado = df.copy()

# Calcula el percentil 99 para margen_bruto en Carabanchel
umbral = df_filtrado[df_filtrado['distrito'] == 'Carabanchel']['margen_bruto'].quantile(0.99)

# Elimina los outliers solo en Carabanchel
df_filtrado = df_filtrado[~((df_filtrado['distrito'] == 'Carabanchel') & (df_filtrado['margen_bruto'] > umbral))]

plt.figure(figsize=(14, 8))
box = sns.boxplot(
    x='margen_bruto', 
    y='distrito', 
    data=df_filtrado, 
    orient='h',
    boxprops=dict(facecolor='skyblue', alpha=0.6, linewidth=2),
    medianprops=dict(color='darkblue', linewidth=2)
)
medianas = df_filtrado.groupby('distrito')['margen_bruto'].median()
for i, distrito in enumerate(medianas.index):
    plt.text(
        medianas[distrito], i, f'{medianas[distrito]:.2f}',
        va='center', ha='left', fontsize=14, fontweight='bold',
        color='black', bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, boxstyle='round,pad=0.2')
    )
plt.xlim(0, df_filtrado['margen_bruto'].max() * 1.1)
plt.xlabel('Margen bruto', fontsize=16)
plt.ylabel('Distrito', fontsize=16)
plt.title('Margen bruto por distrito (sin outlier en Carabanchel)', fontsize=18, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

# Nivel 1: ¿Qué comprar en los siguientes distritos?

Comprar piso completo, menos en Barajas, donde también se recomienda Private room y Shared Room.

In [ ]:
# Crea un df con los registros Centro, Arganzuela, Moncloa - Aravaca y Puente de Vallecas
df_seleccionados = df_filtrado[df_filtrado['distrito'].isin(['Centro', 'Arganzuela', 'Moncloa - Aravaca', 'Tetuán', 'Puente de Vallecas', 'Barajas', 'Villa de Vallecas'])]

# Sobre ese df crea un mapa de calor con el margen bruto
pivot = df_seleccionados.pivot_table(index='distrito', columns='room_type', values='margen_bruto', aggfunc='median')
plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title('Margen bruto por distrito y tipo de habitación')
plt.xlabel('Tipo de habitación')
plt.ylabel('Distrito')
plt.tight_layout()
plt.show()

In [ ]:
# crea un heatmap con el margen_bruto por room_type y bedrooms_disc
pivot = df_seleccionados.pivot_table(index='room_type', columns='bedrooms_disc', values='margen_bruto', aggfunc='median')
plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title('Margen bruto por tipo de habitación y número de dormitorios')
plt.xlabel('Número de dormitorios')
plt.ylabel('Tipo de habitación')
plt.tight_layout()
plt.show()

# Nivel 2: ¿Qué comprar en el resto de distritos?

- 5+ habitaciones en Usera, Moratalaz y Puente de Vallecas
- 3 o +5 habitaciones para Barajas
- 3 habitaciones en Carabanchel
- 2 habitaciones en San Blas
- Investigar Barajas y Usera

En todos ellos para alquilar por piso completo y que se puedan acomodar 4 o más personas.



In [ ]:
# Sobre df filtra solo los que son Entire/room y que no son de distritos Villa de Vallecas, Barajas y Centro
filtro = (
    (df['room_type'] == 'Entire home/apt') &
    (~df['distrito'].isin(['Villa de Vallecas', 'Barajas', 'Centro']))
)
df_filtrado = df[filtro]

In [ ]:
# Lista completa de distritos presentes en el DataFrame original
todos_distritos = sorted(df['distrito'].unique())

# Creamos el pivot asegurando que todos los distritos estén como índice
pivot = df.pivot_table(
    index='distrito',
    columns='bedrooms_disc',
    values='margen_bruto',
    aggfunc='median'
).reindex(todos_distritos)

plt.figure(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title('Margen bruto por distrito y número de dormitorios')
plt.xlabel('Número de dormitorios')
plt.ylabel('Distrito')
plt.tight_layout()
plt.show()

## Nivel 3: Análisis por barrios

In [ ]:
# Agrupa por distrito, neighbourhood, bedrooms_disc, accommodates_disc y calcula la mediana de margen_bruto y estimated_occupancy_l365d
agrupado = df.groupby(['distrito', 'neighbourhood', 'bedrooms_disc', 'accommodates_disc']).agg(
    mediana_margen_bruto=('margen_bruto', 'median'),
    mediana_estimated_occupancy_l365d=('estimated_occupancy_l365d', 'median')
).reset_index()

# ordena por margen_bruto descendente y muestra los 20 primeros
agrupado_ordenado = agrupado.sort_values(by='mediana_margen_bruto', ascending=False).head(20)
agrupado_ordenado

In [ ]:
import folium
from folium.plugins import HeatMap

umbral_margen = 20  # Cambia este valor al margen bruto mínimo que quieras visualizar

df_mapa = df[df['margen_bruto'] >= umbral_margen][['latitude', 'longitude', 'margen_bruto']].dropna()
madrid_coords = [40.4168, -3.7038]
m = folium.Map(location=madrid_coords, zoom_start=11)

head_data = [[row['latitude'], row['longitude'], row['margen_bruto']] for index, row in df_mapa.iterrows()]
HeatMap(head_data, radius=8, blur=15, max_zoom=1).add_to(m)

#from IPython.display import display
#display(m)

# Muestra el mapa
m.save('../entregables/graficos/mapa_rentabilidad.html')

In [ ]:
# Mapa de calor de ocupación con filtro por días
umbral_dias = 250  # Cambia este valor para filtrar por ocupación mínima

df_ocup = df[df['estimated_occupancy_l365d'] >= umbral_dias][['latitude', 'longitude', 'estimated_occupancy_l365d']].dropna()
madrid_coords = [40.4168, -3.7038]
m_ocup = folium.Map(location=madrid_coords, zoom_start=11)

ocup_data = [[row['latitude'], row['longitude'], row['estimated_occupancy_l365d']] for index, row in df_ocup.iterrows()]
HeatMap(ocup_data, radius=8, blur=15, max_zoom=1).add_to(m_ocup)

#from IPython.display import display
# display(m_ocup)

# Guarda el mapa de ocupación
m_ocup.save('../entregables/graficos/mapa_ocupacion.html')

In [ ]:
# Mapa de oportunidades (alta rentabilidad y ocupación)
# Elige los umbrales manualmente
umbral_rent = 15    # Cambia este valor al margen bruto mínimo deseado
umbral_ocup = 250   # Cambia este valor a la ocupación mínima deseada (en días)

df_oportunidades = df[(df['margen_bruto'] >= umbral_rent) & (df['estimated_occupancy_l365d'] >= umbral_ocup)]

m_oportunidades = folium.Map(location=madrid_coords, zoom_start=11)

for _, row in df_oportunidades.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='green',
        fill=True,
        fill_color='orange',
        fill_opacity=0.7,
        popup=f"Rentabilidad: {row['margen_bruto']:.2f}, Ocupación: {row['estimated_occupancy_l365d']:.0f}"
    ).add_to(m_oportunidades)

# Guarda el mapa de oportunidades
m_oportunidades.save('../entregables/graficos/mapa_oportunidades.html')

In [ ]:
# Heatmap de margen bruto solo para Barajas
df_barajas = df[df['distrito'] == 'Barajas'][['latitude', 'longitude', 'margen_bruto']].dropna()
madrid_coords = [40.4668, -3.5778]  # Coordenadas aproximadas del centro de Barajas
m_barajas = folium.Map(location=madrid_coords, zoom_start=13)

barajas_data = [[row['latitude'], row['longitude'], row['margen_bruto']] for index, row in df_barajas.iterrows()]
HeatMap(barajas_data, radius=12, blur=20, max_zoom=1).add_to(m_barajas)

# Guarda el mapa en un archivo HTML
m_barajas.save('../entregables/graficos/mapa_barajas.html')